In [ ]:
# 📌 Setup & Import
import os
import torch
from datetime import datetime
from pathlib import Path
from device_config import DEVICE, info_device
from data_loader import (
    load_mmi_from_cont_mmi,
    load_evac_candidates_geojson,
    load_user_coords,
    load_event_dataset,
    generate_user_coords_from_shp,
    generate_and_save_user_coords_if_needed,
)
from meta_rl_reinforce_baseline import (
    PolicyNetwork,
    adapt_and_log,
    evaluate_model,
    get_adaptive_evac_candidates,
    meta_train,
    save_osrm_cache,
    load_osrm_cache,
)
from config import (
    INNER_STEPS,
    MAX_META_ITER,
    LEARNING_RATE,
    USE_SHP_FOR_USER,
    SHP_PATH,
    N_USER_POINTS,
    USER_JSON_PATH,
)

In [ ]:
# 📌 Informasi device
info_device()

In [ ]:
# 📌 Konfigurasi path & event list
DATA_DIR = "./kota-surabaya"
USER_COORDS_PATH = os.path.join(DATA_DIR, "user_locations.json")
EVENT_LIST = [
    "us6000k49j",
    "us6000mkfz",
    #   "us60005kta",
    #   "usc000nahz",
    #   "usp000ej1c"
]
SAVE_DIR = "./kota-surabaya"
os.makedirs(SAVE_DIR, exist_ok=True)
# load_osrm_cache("./osrm_cache/osrm_cache_baru_shp.json")

In [ ]:
generate_and_save_user_coords_if_needed(SHP_PATH, USER_JSON_PATH, n=N_USER_POINTS)

In [ ]:
# # 📌 Bangun task list dari data event dan SHP user
# def build_task_list(event_list, data_dir):
#     if USE_SHP_FOR_USER:
#         print("📍 Menggunakan SHP untuk generate titik user acak...")
#         user_coords = generate_user_coords_from_shp(SHP_PATH, n=N_USER_POINTS)
#     else:
#         print("📍 Menggunakan JSON titik user tetap...")
#         user_coords = load_user_coords(USER_JSON_PATH)

#     tasks = []
#     for event_id in event_list:
#         try:
#             evac_candidates, mmi_points = load_event_dataset(event_id, data_dir)

#             mmi_coords = torch.tensor(
#                 [[m["lat"], m["lon"]] for m in mmi_points], dtype=torch.float32
#             ).to(DEVICE)
#             mmi_values = torch.tensor(
#                 [m["mmi"] for m in mmi_points], dtype=torch.float32
#             ).to(DEVICE)

#             task = {
#                 "event_id": event_id,
#                 "user_coords": user_coords,
#                 "evac_candidates": evac_candidates,
#                 "mmi_points": mmi_points,
#                 "mmi_coords": mmi_coords,
#                 "mmi_values": mmi_values,
#             }
#             tasks.append(task)
#         except Exception as e:
#             print(f"[SKIP] Event {event_id} gagal dimuat: {e}")

#     return tasks

In [ ]:
# 📌 Bangun task list dari data event + sumber terpisah (evac GeoJSON & cont_mmi JSON)
def build_task_list(event_list, data_dir, evac_geojson_path):
    if USE_SHP_FOR_USER:
        print("📍 Menggunakan SHP untuk generate titik user acak...")
        user_coords = generate_user_coords_from_shp(SHP_PATH, n=N_USER_POINTS)
    else:
        print("📍 Menggunakan JSON titik user tetap...")
        user_coords = load_user_coords(USER_JSON_PATH)

    # Ambil kandidat evakuasi dari GeoJSON sekali (dipakai semua event)
    evac_candidates = load_evac_candidates_geojson(evac_geojson_path)

    tasks = []
    for event_id in event_list:
        try:
            # MMI per event dari file {event_id}-cont_mmi.json
            mmi_json_path = os.path.join(data_dir, f"{event_id}-cont_mmi.json")
            mmi_points = load_mmi_from_cont_mmi(mmi_json_path)

            # Optional: siapkan tensor kalau pipeline adapt/rollout kamu sudah memakai tensor
            mmi_coords = torch.tensor(
                [[m["lat"], m["lon"]] for m in mmi_points], dtype=torch.float32
            ).to(DEVICE)
            mmi_values = torch.tensor(
                [m["mmi"] for m in mmi_points], dtype=torch.float32
            ).to(DEVICE)

            tasks.append(
                {
                    "event_id": event_id,
                    "user_coords": user_coords,
                    "evac_candidates": evac_candidates,
                    "mmi_points": mmi_points,  # tetap disimpan bila fungsi lain perlu
                    "mmi_coords": mmi_coords,  # untuk adapt/rollout
                    "mmi_values": mmi_values,  # untuk adapt/rollout
                }
            )
        except Exception as e:
            print(f"[SKIP] Event {event_id} gagal dimuat: {e}")

    return tasks

In [ ]:
# TASK_LIST = build_task_list(EVENT_LIST, DATA_DIR)

# if USE_SHP_FOR_USER:
#     print("Mode INFERENSI/EVALUASI dengan user acak dari SHP")
# else:
#     print("Mode TRAINING dengan user tetap dari JSON")

In [ ]:
TASK_LIST = build_task_list(
    EVENT_LIST,
    DATA_DIR,
    evac_geojson_path=os.path.join(DATA_DIR, "DATA-WONOCOLO.geojson"),
)

In [ ]:
# 🚀 Meta-Training
meta_model = meta_train(TASK_LIST)

In [ ]:
# 💾 Simpan model
today_str = datetime.today().strftime("%Y-%m-%d")
save_path = Path(SAVE_DIR) / today_str
save_path.mkdir(parents=True, exist_ok=True)
torch.save(meta_model.state_dict(), save_path / "meta_model_final.pt")
print(f"✅ Meta-model disimpan: {save_path/'meta_model_final.pt'}")

In [ ]:
# from datetime import datetime
# from pathlib import Path

# # Tentukan path folder log
# base_dir = Path("train_log")
# today_str = datetime.today().strftime("%Y-%m-%d")
# folder = base_dir / today_str
# folder.mkdir(parents=True, exist_ok=True)

# # Cari nama file log yang belum ada
# base_filename = "rl_logs_meta_event"
# index = 1
# while True:
#     filename = (
#         f"{base_filename}.json" if index == 1 else f"{base_filename}_{index}.json"
#     )
#     output_path = folder / filename
#     if not output_path.exists():
#         break
#     index += 1

In [ ]:
# # 🔍 Evaluasi tiap event + log adaptasi & radius
# for task in TASK_LIST:
#     adapt_and_log(
#         meta_model,
#         task["user_coords"],
#         task["evac_candidates"],
#         task["mmi_points"],
#         task["event_id"],
#         output_path=str(output_path),  # ← ubah dari string ke hasil pencarian file
#     )

In [ ]:
# ✅ Selesai
print("🎉 Evaluasi selesai! Log tersimpan per event.")